# AACAgent Evaluation Notebook

Runs the full multi-turn evaluation pipeline and produces a CSV file that can later be analysed in `metric_evaluation.ipynb`.

## 0. Config

On the **cluster** every variable is injected via `NB_*` environment variables by the `.sh` script.  
On **Colab / local** edit the fallback values directly.

In [ ]:
import os

# ── Environment detection ─────────────────────────────────────────────────────
# NB_IS_COLAB can override auto-detection (useful when nbconvert runs on cluster)
_is_colab_env = os.environ.get('NB_IS_COLAB', '').lower()
if _is_colab_env == 'false':
    IS_COLAB = False
elif _is_colab_env == 'true':
    IS_COLAB = True
else:
    IS_COLAB = 'google.colab' in str(globals().get('__builtins__', ''))

print(f'Running on: {"Colab/local" if IS_COLAB else "Cluster"}')

# ── Core parameters ───────────────────────────────────────────────────────────
# On cluster: read from env; on Colab: use defaults below.
MODELS_RAW     = os.environ.get('NB_MODELS',          'Qwen/Qwen2.5-3B-Instruct')
SPLIT          = os.environ.get('NB_SPLIT',            'both')          # clear | vague | both
N_ROWS_ENV     = os.environ.get('NB_N_ROWS',           '0')             # 0 = full dataset
SEED           = int(os.environ.get('NB_SEED',          '42'))
LOAD_8BIT_ENV  = os.environ.get('NB_LOAD_8BIT',         'False')
MAX_NEW_TOKENS = int(os.environ.get('NB_MAX_NEW_TOKENS', '512'))
LANG_CODE      = os.environ.get('NB_LANG',              'en_eval')

# OUTPUT_CSV: cluster writes eval_hf.csv, Colab writes eval_colab.csv.
# Keeping them separate lets metric_evaluation.ipynb merge them with
# a 'source' label and do cross-environment comparison.
_default_csv = 'eval/results/eval_colab.csv' if IS_COLAB else 'eval/results/eval_hf.csv'
OUTPUT_CSV   = os.environ.get('NB_OUTPUT_CSV', _default_csv)

# On Colab, cap dataset rows automatically to keep execution time reasonable.
# Cap applies only when N_ROWS == 0 (i.e. user has not set an explicit limit).
COLAB_SAMPLE = 200

# Parse compound types
MODELS    = MODELS_RAW.split()
N_ROWS    = int(N_ROWS_ENV)          # 0 = full dataset
LOAD_8BIT = LOAD_8BIT_ENV.lower() == 'true'
HF_DEVICE = 'auto'

print(f'Models        : {MODELS}')
print(f'Split         : {SPLIT}')
print(f'N_rows        : {N_ROWS if N_ROWS > 0 else "full dataset"}')
print(f'Seed          : {SEED}')
print(f'Load 8-bit    : {LOAD_8BIT}')
print(f'Max new tokens: {MAX_NEW_TOKENS}')
print(f'Lang          : {LANG_CODE}')
print(f'Output CSV    : {OUTPUT_CSV}')

## 1. Environment setup

Installs all required packages. On the cluster the venv is already prepared by the `.sh` script so this cell is fast (packages already cached). On Colab it installs from scratch.

In [ ]:
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Core deps always needed
_pip(
    'transformers>=4.40,<5.0',
    'accelerate>=0.29',
    'datasets>=2.19',
    'pandas>=2.0',
    'pyarrow>=14',
    'huggingface_hub>=0.22',
    'tqdm>=4.66',
    'sentencepiece>=0.1.99',
    'protobuf>=3.20',
    'spacy>=3.7',
    'fastmcp',
    'pydantic>=2.0',
    'httpx>=0.24',
    'python-dotenv',
)

if LOAD_8BIT:
    _pip('bitsandbytes>=0.43')

# spaCy model
try:
    import spacy
    spacy.load('en_core_web_sm')
    print('spaCy model en_core_web_sm already present.')
except OSError:
    subprocess.check_call([
        sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'
    ])

print('All packages ready.')

## 2. Path setup

In [ ]:
from pathlib import Path

# Detect project root (works both on Colab after clone and on the cluster).
# Notebook lives at <project_root>/eval/eval.ipynb
_NB_DIR = Path().resolve()    # CWD when jupyter runs the notebook
if (_NB_DIR / 'eval' / 'eval.ipynb').exists():
    # CWD is already the project root (cluster via nbconvert from root)
    PROJECT_ROOT = _NB_DIR
else:
    # CWD is eval/ (interactive run from inside the folder)
    PROJECT_ROOT = _NB_DIR.parent

SRC = PROJECT_ROOT / 'app' / 'src'
APP = PROJECT_ROOT / 'app'

for p in [str(SRC), str(APP)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Resolve output path relative to project root
_out = Path(OUTPUT_CSV)
if not _out.is_absolute():
    _out = PROJECT_ROOT / _out
_out.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV_PATH = _out

EVAL_PARQUET = PROJECT_ROOT / 'eval' / 'eval_filtered.parquet'

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'SRC          : {SRC}')
print(f'Eval parquet : {EVAL_PARQUET}  exists={EVAL_PARQUET.exists()}')
print(f'Output CSV   : {OUTPUT_CSV_PATH}')

## 3. Imports

In [ ]:
import csv
import logging
import time
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Optional

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Logging ───────────────────────────────────────────────────────────────────
try:
    from logs.logging_config import setup_logging
    setup_logging()
except Exception:
    logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s: %(message)s')

logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('urllib3').setLevel(logging.WARNING)

# ── Project imports ───────────────────────────────────────────────────────────
from config import AGENT_DEFAULT_MODEL
from agent.agent import AACAgent, EvalContext
from agent.hf_agent import HFAACAgent
from agent.session import SessionMemory
from agent.resolve import RESOLVE_METHODS
from mcp_server.models import Pictogram, Keyword
from mcp_server.tools.arasaac import get_pictogram_metadata
import mcp_server.tools.arasaac as _arasaac_mod

# Override the language used by all ARASAAC tool calls
_arasaac_mod.LANG = LANG_CODE  # type: ignore[attr-defined]

print('Imports OK.')
print(f'CUDA available: {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  ({p.total_memory/1e9:.1f} GB)')

## 4. Constants

In [ ]:
DATE_BASE  = '2025-01-15'
SLOT_TIMES = {
    'morning':   '09:00',
    'afternoon': '15:00',
    'evening':   '19:00',
    'night':     '22:00',
}

OVERLAP_LEVELS = ['synset', 'category', 'keyword', 'tag']

CSV_COLUMNS = [
    'model', 'row_idx', 'split', 'turn_pos', 'n_turns_total', 'caregiver_input', 'concept',
    'gold_id', 'all_gold_ids', 'predicted_ids', 'gold_in_candidates', 'hit',
    'n_candidates', 'window_len', 'called_get_time', 'called_get_schedule',
    'overlap_level',
    'resolve_method',   # exact | lemma | hyphen | lemma_alt | token | none
    'resolve_queries',  # list of keyword strings passed to search_pictograms
    'plan_method',      # llm | fallback_spacy | fallback_empty
    'synset_added',     # pictograms added by synset expansion this turn
    'fresh_count',      # pictograms in window from fresh pool (not stale padding)
    'planner_had_gold_concept',  # True iff planner generated exactly the gold concept this turn
    'input_triggered_tools',     # True only at turn_pos==0 (caregiver gave real input)
]

## 5. Dataset loading

In [ ]:
print(f'Loading dataset from {EVAL_PARQUET} …')
df_full = pd.read_parquet(EVAL_PARQUET)
print(f'Full dataset: {len(df_full):,} rows × {df_full.shape[1]} cols')

if IS_COLAB:
    # On Colab: always cap to COLAB_SAMPLE unless an explicit N_ROWS was set.
    # This prevents accidentally running the full dataset on a free-tier T4.
    _cap = N_ROWS if N_ROWS > 0 else COLAB_SAMPLE
    df   = df_full.sample(min(_cap, len(df_full)), random_state=SEED).reset_index(drop=True)
    print(f'Colab sample: {len(df)} rows (cap={_cap}, seed={SEED})')
else:
    # Cluster: use full dataset unless N_ROWS explicitly set.
    if N_ROWS > 0:
        df = df_full.sample(N_ROWS, random_state=SEED).reset_index(drop=True)
        print(f'Sampled: {len(df)} rows (seed={SEED})')
    else:
        df = df_full
        print('Using full dataset (cluster mode)')

print(f'Effective rows: {len(df)}')
df.head(3)

## 6. Helper functions

In [ ]:
# ── Gold metadata cache ───────────────────────────────────────────────────────

_gold_cache: dict[int, dict] = {}

def get_gold_meta(pic_id: int) -> dict:
    """Fetch and cache pictogram metadata for a gold ID."""
    k = int(pic_id)
    if k not in _gold_cache:
        try:
            _gold_cache[k] = get_pictogram_metadata(pictogram_id=k, lang=LANG_CODE)
        except Exception:
            _gold_cache[k] = {}
    return _gold_cache[k]


def gold_as_pictogram(pic_id: int, concept: str) -> Pictogram:
    """Return a Pictogram for a gold ID (fallback to minimal stub if metadata missing)."""
    meta = get_gold_meta(pic_id)
    if meta:
        try:
            return Pictogram.model_validate(meta)
        except Exception:
            pass
    return Pictogram(id=int(pic_id), keywords=[Keyword(type=2, keyword=concept)])

In [ ]:
# ── Semantic overlap ──────────────────────────────────────────────────────────

def _pic_to_dict(p) -> dict:
    if isinstance(p, dict):          return p
    if hasattr(p, 'model_dump'):     return p.model_dump()
    return vars(p)

def _pic_synsets(d: dict)    -> set: return set(d.get('synsets') or [])
def _pic_categories(d: dict) -> set: return set(d.get('categories') or [])
def _pic_tags(d: dict)       -> set: return set(d.get('tags') or [])

def _pic_keywords(d: dict) -> set:
    out: set[str] = set()
    for kw in d.get('keywords') or []:
        if isinstance(kw, dict):
            out.add(kw.get('keyword', '').lower())
        elif hasattr(kw, 'keyword'):
            out.add(kw.keyword.lower())
    return out - {''}


def semantic_overlap_level(pred, gold_meta: dict) -> Optional[str]:
    """Return the best overlap level between a predicted pictogram and gold metadata."""
    p = _pic_to_dict(pred)
    ps, gs = _pic_synsets(p), _pic_synsets(gold_meta)
    if ps and gs and ps & gs:                            return 'synset'
    if _pic_categories(p) & _pic_categories(gold_meta): return 'category'
    if _pic_keywords(p)   & _pic_keywords(gold_meta):   return 'keyword'
    if _pic_tags(p)        & _pic_tags(gold_meta):       return 'tag'
    return None


def window_best_overlap(window: list, gold_meta: dict) -> Optional[str]:
    """Return the best overlap level found across the entire predicted window."""
    best_idx = len(OVERLAP_LEVELS)
    for pic in window:
        level = semantic_overlap_level(pic, gold_meta)
        if level is not None:
            idx = OVERLAP_LEVELS.index(level)
            if idx < best_idx:
                best_idx = idx
            if best_idx == 0:
                break
    return OVERLAP_LEVELS[best_idx] if best_idx < len(OVERLAP_LEVELS) else None

In [ ]:
# ── Mock tool helpers ─────────────────────────────────────────────────────────

def _make_mock_time(time_of_day: str) -> dict:
    """Build a mock get_time() response from a slot name."""
    hhmm = SLOT_TIMES.get(str(time_of_day).lower(), '09:00')
    return {'current_dt': f'{DATE_BASE}T{hhmm}:00', 'time_of_day': time_of_day}


def _make_mock_schedule(schedule_arr) -> list:
    """Normalise a schedule array (any format) to a list of plain dicts."""
    if schedule_arr is None:
        return []
    out = []
    for ev in schedule_arr:
        if hasattr(ev, 'items'):     out.append(dict(ev))
        elif hasattr(ev, '_asdict'): out.append(ev._asdict())
        else:
            out.append({k: getattr(ev, k, None)
                        for k in ('title', 'start_time', 'location', 'description')})
    return out

In [ ]:
# ── Teacher forcing ───────────────────────────────────────────────────────────

def teacher_force(agent: AACAgent, gold_id: int, concept: str) -> None:
    """Replace the last memory turn with the gold pictogram.

    This simulates the caregiver manually finding and selecting the correct
    pictogram when the agent missed it, keeping multi-turn sequences valid.
    """
    if not agent.memory.turns:
        return
    last = agent.memory.turns[-1]
    for t in last.topics:
        agent.memory.topic_frequency[t] = max(0, agent.memory.topic_frequency.get(t, 0) - 1)
    gold_pic    = gold_as_pictogram(gold_id, concept)
    gold_topics = SessionMemory.extract_topics([gold_pic])
    last.pictograms = [gold_pic]
    last.topics     = gold_topics
    for t in gold_topics:
        agent.memory.topic_frequency[t] = agent.memory.topic_frequency.get(t, 0) + 1

In [ ]:
# ── Resolve info extractor ────────────────────────────────────────────────────

def get_resolve_info(agent: AACAgent, concept: str) -> tuple[str, list[str], bool]:
    """Extract resolve info for a gold concept from last_resolve_info.

    Returns
    -------
    method            : resolve step label ('exact', 'lemma', …, 'none')
    queries           : keyword strings passed to search_pictograms for this concept
    planner_had_gold  : True iff the planner generated exactly this concept during _plan().

    For turn_pos > 0 the planner receives empty input and infers concepts from
    history — it typically does NOT generate the gold concept by name.  In that
    case planner_had_gold=False and method='none' is expected, not a failure of
    the resolve step.  Filter by planner_had_gold=True when computing meaningful
    resolve-method distributions.
    """
    for entry in agent.last_resolve_info:
        if entry['concept'] == concept:
            return entry['method'], entry['queries'], True
    return 'none', [], False

In [ ]:
# ── Multi-turn runner ─────────────────────────────────────────────────────────

def run_multi_turn(agent: AACAgent, row: 'pd.Series', split: str) -> list[dict]:
    """Run all turns for one dataset row with teacher forcing.

    Parameters
    ----------
    agent : AACAgent (or HFAACAgent) instance — reused across rows.
    row   : One row from the eval dataframe.
    split : 'clear' or 'vague'.

    Returns
    -------
    List of result dicts (one per turn), ready to be written to CSV.
    """
    concepts  = list(row['concept'])
    gold_ids  = [int(g) for g in row['best_id']]
    n_turns   = len(concepts)
    caregiver = row['caregiver_clear'] if split == 'clear' else row['caregiver_vague']

    if split == 'vague':
        mock_time     = _make_mock_time(row['time_of_day'])
        mock_schedule = _make_mock_schedule(row.get('schedule'))
    else:
        mock_time, mock_schedule = None, []

    agent.reset_session()
    results: list[dict] = []

    for turn_pos, (concept, gold_id) in enumerate(zip(concepts, gold_ids)):
        ec     = EvalContext(mock_time=mock_time, mock_schedule=mock_schedule)
        raw_in = caregiver if turn_pos == 0 else ''

        window     = agent.run(raw_in, eval_ctx=ec)
        candidates = list(agent.last_candidates)

        predicted_ids  = [p.id for p in window]
        gold_in_cands  = gold_id in {p.id for p in candidates}
        hit            = gold_id in set(predicted_ids)
        gold_meta      = get_gold_meta(gold_id)
        overlap_level  = window_best_overlap(window, gold_meta)
        resolve_method, resolve_queries, planner_had_gold = get_resolve_info(agent, concept)

        results.append({
            'row_idx':             row.name,
            'split':               split,
            'turn_pos':            turn_pos,
            'n_turns_total':       n_turns,
            'caregiver_input':     raw_in,
            'concept':             concept,
            'gold_id':             gold_id,
            'all_gold_ids':        gold_ids,
            'predicted_ids':       predicted_ids,
            'gold_in_candidates':  gold_in_cands,
            'hit':                 hit,
            'n_candidates':        len(candidates),
            'window_len':          len(window),
            'called_get_time':     'get_time'     in ec.tool_calls,
            'called_get_schedule': 'get_schedule' in ec.tool_calls,
            'overlap_level':       overlap_level,
            'resolve_method':           resolve_method,
            'resolve_queries':          resolve_queries,
            'plan_method':              agent.last_plan_method,
            'synset_added':             agent.last_synset_added,
            'fresh_count':              agent.last_fresh_count,
            'planner_had_gold_concept': planner_had_gold,
            'input_triggered_tools':    turn_pos == 0,
        })

        teacher_force(agent, gold_id, concept)

    return results

In [ ]:
# ── Incremental CSV writer ────────────────────────────────────────────────────

class IncrementalCSV:
    """Append-mode CSV writer with buffering. Creates header on first write."""

    def __init__(self, path: Path) -> None:
        self.path    = path
        self._is_new = not path.exists()
        self._buffer: list[dict] = []

    def add(self, rows: list[dict]) -> None:
        self._buffer.extend(rows)

    def flush(self) -> None:
        if not self._buffer:
            return
        mode = 'w' if self._is_new else 'a'
        with open(self.path, mode, newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction='ignore')
            if self._is_new:
                writer.writeheader()
                self._is_new = False
            writer.writerows(self._buffer)
        self._buffer.clear()


# ── Progress tracker ──────────────────────────────────────────────────────────

class ProgressTracker:
    """Tracks hit-rate and ETA across all splits for one model."""

    def __init__(self, total: int, splits: list[str]) -> None:
        self.total    = total
        self.n_done   = 0
        self.n_errors = 0
        self.hits: dict[str, list[bool]] = {s: [] for s in splits}
        self._t0      = time.monotonic()

    def record(self, split: str, row_results: list[dict]) -> None:
        self.n_done += 1
        for r in row_results:
            self.hits[split].append(bool(r['hit']))

    def record_error(self) -> None:
        self.n_done  += 1
        self.n_errors += 1

    def print_progress(self) -> None:
        elapsed  = time.monotonic() - self._t0
        avg_s    = elapsed / max(self.n_done, 1)
        eta_str  = str(timedelta(seconds=int(avg_s * max(self.total - self.n_done, 0))))
        hit_strs = [f'{s}={sum(h)/len(h):.3f}' for s, h in self.hits.items() if h]
        w = len(str(self.total))
        print(
            f'  [{self.n_done:>{w}}/{self.total}]'
            f'  hit@window: {" ".join(hit_strs) or "—"}'
            f'  ETA {eta_str}'
            f'  errors={self.n_errors}',
            flush=True,
        )

## 7. Evaluation loop

In [ ]:
LOG_EVERY  = 10
SAVE_EVERY = 10

splits = ['clear', 'vague'] if SPLIT == 'both' else [SPLIT]

csv_writer = IncrementalCSV(OUTPUT_CSV_PATH)
total_t0   = time.monotonic()

for model_name in MODELS:
    print(f'\n{"━"*70}')
    print(f'  MODEL: {model_name}')
    print(f'{"━"*70}')

    agent = HFAACAgent(
        model             = model_name,
        hf_device         = HF_DEVICE,
        hf_load_in_8bit   = LOAD_8BIT,
        hf_max_new_tokens = MAX_NEW_TOKENS,
        lang              = LANG_CODE,
    )

    # One tracker per model — resets counters and ETA for each model.
    total_work = len(df) * len(splits)
    tracker    = ProgressTracker(total=total_work, splits=splits)

    for split in splits:
        print(f'\n  ── split: {split.upper()} ──')

        for i, (_, row) in enumerate(df.iterrows()):
            try:
                row_results = run_multi_turn(agent, row, split)
                for r in row_results:
                    r['model'] = model_name
                csv_writer.add(row_results)
                tracker.record(split, row_results)
            except Exception as exc:
                tracker.record_error()
                print(f'  [ERROR] row={row.name}  model={model_name}: {exc}', flush=True)

            if tracker.n_done % SAVE_EVERY == 0:
                csv_writer.flush()

            if tracker.n_done % LOG_EVERY == 0 or tracker.n_done == total_work:
                tracker.print_progress()

    csv_writer.flush()
    agent.unload()
    print(f'  Model {model_name!r} done — GPU memory freed.')

csv_writer.flush()
elapsed = time.monotonic() - total_t0
print(f'\n{"═"*70}')
print(f'  All models done in {timedelta(seconds=int(elapsed))}')
print(f'  Output: {OUTPUT_CSV_PATH}')
print(f'{"═"*70}')

## 8. Sanity check on the output

In [ ]:
import ast

res = pd.read_csv(OUTPUT_CSV_PATH)
for col in ('predicted_ids', 'all_gold_ids', 'resolve_queries'):
    if col in res.columns:
        res[col] = res[col].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )

print(f'CSV rows: {len(res):,}')
print(f'Columns : {list(res.columns)}')
print(f'Models  : {res["model"].unique().tolist()}')
print(f'Splits  : {res["split"].unique().tolist()}')
print(f'Overall hit@window: {res["hit"].mean():.3f}')
res.head(5)